In [1]:
"""
train_xceptionnet.py
=====================
Self-contained training script untuk XceptionNet pada FaceForensics++ C23.
Sudah include: config, kaggle download, dataset loader, training loop, save checkpoint.

Jalankan:
  pip install kagglehub timm torch torchvision opencv-python scikit-learn tqdm pandas
  python train_xceptionnet.py

Output:
  checkpoints/xceptionnet_best.pth
  checkpoints/xceptionnet_last.pth
  checkpoints/xceptionnet_meta.json
"""

# ══════════════════════════════════════════════════════════════
# 0. IMPORTS
# ══════════════════════════════════════════════════════════════
import cv2
import json
import logging
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

import kagglehub
import torch
import numpy as np

# Tambahkan baris ini tepat di bawah import
torch.serialization.add_safe_globals([np.core.multiarray.scalar])

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════
# 1. CONFIG
# ══════════════════════════════════════════════════════════════
CFG = {
    "kaggle_dataset":   "faceforensics-extracted-dataset-c23",
    "csv_filename":     "FF++C32-All-frames.csv",
    "img_size":         299,
    "label_col":        "label",
    "filepath_col":     "filepath",

    "real_label":       "Original",
    "fake_labels": [
        "Deepfakes",
        "Face2Face",
        "FaceShifter",
        "FaceSwap",
        "NeuralTextures",
    ],
    "batch_size":       32,
    "num_epochs":       30,
    "lr":               1e-4,
    "weight_decay":     1e-5,
    "patience":         7,
    "val_split":        0.15,
    "test_split":       0.15,
    "random_seed":      42,
    "num_workers":      4,
    "label_smoothing":  0.1,
    "checkpoint_dir":   "checkpoints",
    "model_name":       "xceptionnet",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(CFG["random_seed"])
torch.manual_seed(CFG["random_seed"])
Path(CFG["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(f"train_{CFG['model_name']}.log"),
    ],
)
log = logging.getLogger(__name__)
log.info(f"Device : {DEVICE}")
log.info(f"Model  : XceptionNet | img_size={CFG['img_size']}")


# ══════════════════════════════════════════════════════════════
# 2. KAGGLE DATASET LOADER
# ══════════════════════════════════════════════════════════════

def download_and_load_csv() -> tuple[Path, pd.DataFrame]:
    log.info("Menghubungkan ke Kaggle...")
    root = Path(kagglehub.dataset_download(CFG["kaggle_dataset"]))
    log.info(f"Dataset root: {root}")
    csv_candidates = list(root.rglob(CFG["csv_filename"]))
    if not csv_candidates:
        csv_candidates = list(root.rglob("*.csv"))

    if not csv_candidates:
        raise FileNotFoundError(f"Tidak ada CSV di {root}")
    csv_path = csv_candidates[0]
    log.info(f"Memuat CSV: {csv_path}")
    df = pd.read_csv(csv_path)
    log.info(f"CSV shape: {df.shape}")
    log.info(f"Kolom    : {list(df.columns)}")

    return root, df


def build_dataframe(root: Path, df: pd.DataFrame) -> pd.DataFrame:
    """
    Versi cepat untuk Kaggle: Menggabungkan root dataset 
    langsung dengan nama file dari CSV.
    """
    log.info("Resolving path gambar (Fast Mode)...")
    df = df.copy()

    # Kita asumsikan gambar ada di dalam subfolder 'frames' atau langsung di root
    # Kebanyakan dataset ekstraksi FF++ di Kaggle punya struktur folder tertentu.
    # Kita buat fungsi simpel yang tidak pakai rglob.
    
    def fast_resolve(fp):
        filename = Path(fp).name
        # Kaggle dataset biasanya meletakkan gambar di subfolder, 
        # kita coba cari manual sekali saja jalurnya
        return str(root / "all_frames" / filename) # Sesuaikan "all_frames" jika berbeda

    # Jika kamu tidak yakin nama subfoldernya, gunakan mapping satu kali:
    log.info("Memetakan lokasi file...")
    # Scan seluruh file di dataset HANYA SEKALI
    file_map = {p.name: str(p) for p in root.rglob('*.jpg')} 
    
    df["abs_path"] = df[CFG["filepath_col"]].apply(lambda x: file_map.get(Path(x).name, ""))
    
    # Hapus data yang path-nya tidak ditemukan agar tidak error saat training
    df = df[df["abs_path"] != ""].reset_index(drop=True)
    
    # ── Binary label ──────────────────────────────────────────
    df["binary_label"] = df[CFG["label_col"]].apply(
        lambda x: 0 if str(x).strip() == CFG["real_label"] else 1
    )
    df["manipulation_type"] = df[CFG["label_col"]].apply(str)

    log.info(f"Total gambar valid: {len(df)}")
    return df


# ══════════════════════════════════════════════════════════════
# 3. PYTORCH DATASET & TRANSFORMS
# ══════════════════════════════════════════════════════════════

TRAIN_TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

EVAL_TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])


class FF_Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["abs_path"])
        if img is None:
            img = np.zeros((CFG["img_size"], CFG["img_size"], 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        return img, int(row["binary_label"])


def make_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    labels  = df["binary_label"].values
    counts  = np.bincount(labels)
    weights = 1.0 / counts[labels]
    return WeightedRandomSampler(
        torch.DoubleTensor(weights), len(weights), replacement=True
    )


def build_dataloaders(df: pd.DataFrame):
    """Split train/val/test dan buat DataLoader."""
    train_df, temp_df = train_test_split(
        df, test_size=CFG["val_split"] + CFG["test_split"],
        stratify=df["binary_label"], random_state=CFG["random_seed"]
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=CFG["test_split"] / (CFG["val_split"] + CFG["test_split"]),
        stratify=temp_df["binary_label"], random_state=CFG["random_seed"]
    )

    log.info(f"\nSplit — Train:{len(train_df)} | Val:{len(val_df)} | Test:{len(test_df)}")

    sampler = make_weighted_sampler(train_df)

    train_loader = DataLoader(
        FF_Dataset(train_df, TRAIN_TRANSFORM),
        batch_size=CFG["batch_size"], sampler=sampler,
        num_workers=CFG["num_workers"], pin_memory=True,
    )
    val_loader = DataLoader(
        FF_Dataset(val_df, EVAL_TRANSFORM),
        batch_size=CFG["batch_size"], shuffle=False,
        num_workers=CFG["num_workers"], pin_memory=True,
    )
    test_loader = DataLoader(
        FF_Dataset(test_df, EVAL_TRANSFORM),
        batch_size=CFG["batch_size"], shuffle=False,
        num_workers=CFG["num_workers"], pin_memory=True,
    )

    return train_loader, val_loader, test_loader, test_df


# ══════════════════════════════════════════════════════════════
# 4. MODEL — XCEPTIONNET
# ══════════════════════════════════════════════════════════════

def build_model() -> nn.Module:
    try:
        import timm
        base = timm.create_model("xception", pretrained=True, num_classes=0)
        in_features = base.num_features   # 2048

        # Freeze semua kecuali block 12-14 dan head
        for name, param in base.named_parameters():
            freeze = True
            for unfreeze_kw in ["block12", "block13", "block14", "act4", "conv4"]:
                if unfreeze_kw in name:
                    freeze = False
                    break
    class XceptionNet(nn.Module):
        def __init__(self, backbone, in_features):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(
                nn.Dropout(p=0.5),
                nn.Linear(in_features, 256),
                nn.ReLU(),
                nn.Dropout(p=0.3),
                nn.Linear(256, 2),
            )
            self.gradcam_target_layer = backbone.act4
        def forward(self, x):
            feat = self.backbone(x)
            return self.head(feat)
    model = XceptionNet(base, in_features)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"XceptionNet — Trainable params: {n_train:,}")
    return model.to(DEVICE)


# ══════════════════════════════════════════════════════════════
# 5. TRAINING LOOP
# ══════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = correct = total = 0

    for imgs, labels in tqdm(loader, desc="  Train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []

    for imgs, labels in tqdm(loader, desc="  Eval ", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        probs       = F.softmax(logits, dim=1)[:, 1]
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return total_loss / total, correct / total, auc


def run_training(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG["lr"], weight_decay=CFG["weight_decay"]
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["num_epochs"], eta_min=1e-6
    )
    scaler    = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

    best_auc     = 0.0
    patience_ctr = 0
    ckpt_dir     = Path(CFG["checkpoint_dir"])
    best_path    = ckpt_dir / f"{CFG['model_name']}_best.pth"
    last_path    = ckpt_dir / f"{CFG['model_name']}_last.pth"

    log.info(f"\n{'Epoch':>6} | {'TrainLoss':>9} | {'ValLoss':>7} | {'ValAcc':>6} | {'ValAUC':>7}")
    log.info("-" * 52)

    for epoch in range(1, CFG["num_epochs"] + 1):
        tr_loss, tr_acc           = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion)
        scheduler.step()

        log.info(f"  {epoch:>4}  | {tr_loss:>9.4f} | {val_loss:>7.4f} | {val_acc:>6.3f} | {val_auc:>7.4f}")

        # Save last
        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "val_auc": val_auc, "val_acc": val_acc, "cfg": CFG,
        }, last_path)
        imgs = []
        labels = []

        with torch.amp.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        if val_auc > best_auc:
            best_auc     = val_auc
            patience_ctr = 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "val_auc": val_auc, "val_acc": val_acc, "cfg": CFG,
            }, best_path)
            log.info(f"  ✓ Best checkpoint (val_auc={val_auc:.4f})")
        else:
            patience_ctr += 1
            if patience_ctr >= CFG["patience"]:
                log.info(f"\n  Early stopping di epoch {epoch}")
                break

    log.info(f"\n  Best Val AUC : {best_auc:.4f}")
    log.info(f"  Checkpoint   : {best_path}")
    return best_path


# ══════════════════════════════════════════════════════════════
# 6. TEST EVALUATION & SAVE META
# ══════════════════════════════════════════════════════════════

@torch.no_grad()
def test_and_save_meta(model, test_loader, test_df: pd.DataFrame, best_path: Path):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for imgs, labels in tqdm(test_loader, desc="Testing"):
        imgs   = imgs.to(DEVICE)
        logits = model(imgs)
        probs  = F.softmax(logits, dim=1)[:, 1]
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    log.info("\n" + "═"*52)
    log.info("  TEST RESULTS — XceptionNet")
    log.info("═"*52)
    log.info(f"  Accuracy : {acc:.4f}")
    log.info(f"  ROC-AUC  : {auc:.4f}")
    log.info(f"\n{classification_report(all_labels, all_preds, target_names=['real','fake'])}")

    # Simpan test_df untuk dipakai xai_evaluate.py
    test_csv = Path(CFG["checkpoint_dir"]) / f"{CFG['model_name']}_test_frames.csv"
    test_df.to_csv(test_csv, index=False)

    # Simpan metadata
    meta = {
        "model_name":           "XceptionNet",
        "checkpoint":           str(best_path),
        "test_frames_csv":      str(test_csv),
        "img_size":             CFG["img_size"],
        "normalize_mean":       [0.5, 0.5, 0.5],
        "normalize_std":        [0.5, 0.5, 0.5],
        "num_classes":          2,
        "test_acc":             acc,
        "test_auc":             auc,
        "gradcam_layer":        "backbone.act4",
    }
    meta_path = Path(CFG["checkpoint_dir"]) / f"{CFG['model_name']}_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    log.info(f"  Meta disimpan: {meta_path}")
    log.info(f"  Test CSV    : {test_csv}")

2026-05-01 03:11:55,283 [INFO] Device : cuda
2026-05-01 03:11:55,284 [INFO] Model  : XceptionNet | img_size=299


In [2]:
log.info("═"*52)
log.info("  XceptionNet — Training Pipeline")
log.info("═"*52)

root, raw_df              = download_and_load_csv()
df                        = build_dataframe(root, raw_df)
train_loader, val_loader, test_loader, test_df = build_dataloaders(df)
model                     = build_model()
best_path                 = run_training(model, train_loader, val_loader)



2026-05-01 03:11:55,296 [INFO] ════════════════════════════════════════════════════
2026-05-01 03:11:55,297 [INFO]   XceptionNet — Training Pipeline
2026-05-01 03:11:55,299 [INFO] ════════════════════════════════════════════════════
2026-05-01 03:11:55,301 [INFO] Menghubungkan ke Kaggle...
2026-05-01 03:11:55,384 [INFO] Dataset root: /kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23
2026-05-01 03:13:08,606 [INFO] Memuat CSV: /kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23/FF++C32-All-frames.csv
2026-05-01 03:13:08,733 [INFO] CSV shape: (30000, 5)
2026-05-01 03:13:08,733 [INFO] Kolom    : ['filename', 'label', 'features', 'filepath', 'label_id']
2026-05-01 03:13:08,734 [INFO] Resolving path gambar (Fast Mode)...
2026-05-01 03:13:08,736 [INFO] Memetakan lokasi file...
2026-05-01 03:13:21,722 [INFO] Total gambar valid: 30000
2026-05-01 03:13:21,747 [INFO] 
Split — Train:21000 | Val:4500 | Test:4500
2026-05-01 03:13:26,037 [INFO] Loading pret

Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth


2026-05-01 03:13:26,978 [INFO] XceptionNet — Trainable params: 5,724,194
2026-05-01 03:13:27,297 [INFO] 
 Epoch | TrainLoss | ValLoss | ValAcc |  ValAUC
2026-05-01 03:13:27,298 [INFO] ----------------------------------------------------
2026-05-01 03:17:58,313 [INFO]      1  |    0.6826 |  0.6876 |  0.552 |  0.6741
2026-05-01 03:17:58,552 [INFO]   ✓ Best checkpoint (val_auc=0.6741)
2026-05-01 03:22:05,564 [INFO]      2  |    0.5898 |  0.5815 |  0.700 |  0.7713
2026-05-01 03:22:06,068 [INFO]   ✓ Best checkpoint (val_auc=0.7713)
2026-05-01 03:26:14,022 [INFO]      3  |    0.5051 |  0.5435 |  0.758 |  0.8293
2026-05-01 03:26:14,397 [INFO]   ✓ Best checkpoint (val_auc=0.8293)
2026-05-01 03:30:23,855 [INFO]      4  |    0.4513 |  0.4874 |  0.800 |  0.8574
2026-05-01 03:30:24,242 [INFO]   ✓ Best checkpoint (val_auc=0.8574)
2026-05-01 03:34:45,958 [INFO]      5  |    0.4169 |  0.4265 |  0.859 |  0.8830
2026-05-01 03:34:46,353 [INFO]   ✓ Best checkpoint (val_auc=0.8830)
2026-05-01 03:39:10,942

In [4]:
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])

test_and_save_meta(model, test_loader, test_df, best_path)

log.info("\n  Selesai. Jalankan xai_evaluate.py untuk analisis XAI.")

Testing: 100%|██████████| 141/141 [00:58<00:00,  2.42it/s]
2026-05-01 05:31:36,446 [INFO] 
════════════════════════════════════════════════════
2026-05-01 05:31:36,447 [INFO]   TEST RESULTS — XceptionNet
2026-05-01 05:31:36,448 [INFO] ════════════════════════════════════════════════════
2026-05-01 05:31:36,449 [INFO]   Accuracy : 0.9567
2026-05-01 05:31:36,450 [INFO]   ROC-AUC  : 0.9697
2026-05-01 05:31:36,463 [INFO] 
              precision    recall  f1-score   support

        real       0.89      0.84      0.87       750
        fake       0.97      0.98      0.97      3750

    accuracy                           0.96      4500
   macro avg       0.93      0.91      0.92      4500
weighted avg       0.96      0.96      0.96      4500

2026-05-01 05:31:36,514 [INFO]   Meta disimpan: checkpoints/xceptionnet_meta.json
2026-05-01 05:31:36,515 [INFO]   Test CSV    : checkpoints/xceptionnet_test_frames.csv
2026-05-01 05:31:36,516 [INFO] 
  Selesai. Jalankan xai_evaluate.py untuk analisis

In [6]:
import os

# Cari semua file .pth di seluruh sistem
for root, dirs, files in os.walk('/kaggle'):
    for f in files:
        if f.endswith('.pth') or f.endswith('.csv') or f.endswith('.json'):
            print(os.path.join(root, f))

/kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23/FF++C32-All-frames.csv
/kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23/CSVs/FaceSwap.csv
/kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23/CSVs/FaceShifter.csv
/kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23/CSVs/Face2Face.csv
/kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23/CSVs/Deepfakes.csv
/kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23/CSVs/Original.csv
/kaggle/input/datasets/fatimahirshad/faceforensics-extracted-dataset-c23/CSVs/NeuralTextures.csv
/kaggle/working/checkpoints/xceptionnet_last.pth
/kaggle/working/checkpoints/xceptionnet_test_frames.csv
/kaggle/working/checkpoints/xceptionnet_meta.json
/kaggle/working/checkpoints/xceptionnet_best.pth


In [7]:
print("Isi folder checkpoints:")
print(os.listdir('checkpoints'))

Isi folder checkpoints:
['xceptionnet_last.pth', 'xceptionnet_test_frames.csv', 'xceptionnet_meta.json', 'xceptionnet_best.pth']


In [8]:
import shutil
shutil.make_archive('xceptionnet_checkpoint', 'zip', 'checkpoints')

from IPython.display import FileLink
FileLink('xceptionnet_checkpoint.zip')

/kaggle/working/xceptionnet_checkpoint.zip